In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
FOLDER = '/content/drive/MyDrive/CPI_Research'
os.makedirs(FOLDER, exist_ok=True)
print("✅ Drive mounted:", os.listdir(FOLDER))

Mounted at /content/drive
✅ Drive mounted: ['bangladesh_MASTER_dataset.csv', 'lasso_selected_features.csv', 'session1_orders.json', 'fig_session1_predictions.png', 'fig_session2_predictions.png', 'FINAL_scoreboard.csv', 'fig_scoreboard.png', 'fig_eda_series.png', 'fig_correlation.png', 'FINAL_forecast_2026_2027.csv', 'FINAL_forecast_chart.png', 'all_model_results.csv', 'multiseed_results.csv', 'session5_leakage_ablation.py', 'session5_leakage_ablation_results.csv', 'session5_error_vectors.npz', 'session5b_warmup_diagnostic.py', 'session5b_warmup_sensitivity.csv', 'session5c_ablation_matched.py', 'session5c_ablation_results.csv', 'session5c_error_vectors.npz', 'session5d_multiseed_ablation.py', 'session5d_multiseed_ablation.csv', 'Leakage Ablation Runbook_session5.ipynb']


In [15]:
import numpy as np, pandas as pd, itertools, json, warnings, time, io, os
warnings.filterwarnings('ignore')
import requests
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy import stats

SEED = 42; np.random.seed(SEED)

COUNTRIES = {
    'BGD': 'Bangladesh',    # control
    'IND': 'India',
    'PAK': 'Pakistan',
    'LKA': 'Sri Lanka',
    'IDN': 'Indonesia',
    'PHL': 'Philippines',
}
TRAIN_END  = '2020-12-01'
TEST_START = '2021-01-01'
TEST_END   = '2026-04-01'
MIN_TRAIN  = 120
MIN_TEST   = 12
RUN_LSTM   = True        # START WITH False for the fast pass
LSTM_SEEDS = [42, 7, 13, 21, 99, 123, 256, 314, 777, 2024]
print('Config ready:', ', '.join(COUNTRIES.values()))

Config ready: Bangladesh, India, Pakistan, Sri Lanka, Indonesia, Philippines


In [3]:
def rmse(e): return float(np.sqrt(np.mean(e**2)))
def mae(e):  return float(np.mean(np.abs(e)))
def mape(e, actual): return float(np.mean(np.abs(e/actual))*100)

def naive_error_vectors(y, test_start, test_end):
    test = y.loc[test_start:test_end]; idx = test.index
    full = y.loc[:test_end]; vals = full.values
    p0 = full.index.get_loc(idx[0]); n = len(idx); e = {}
    e['RW']       = np.array([vals[p0+i] - vals[p0+i-1] for i in range(n)])
    e['RW+drift'] = np.array([vals[p0+i] - (vals[p0+i-1] + np.mean(np.diff(vals[:p0+i]))) for i in range(n)])
    e['SN']       = np.array([vals[p0+i] - vals[p0+i-12] for i in range(n)])
    e['SN+drift'] = np.array([vals[p0+i] - (vals[p0+i-12] + 12*np.mean(np.diff(vals[:p0+i]))) for i in range(n)])
    return e, test.values

def dm_test(e1, e2, h=1):
    d = e1**2 - e2**2; T = len(d); dbar = d.mean()
    lrv = np.sum((d - dbar)**2) / T
    for k in range(1, h):
        lrv += 2*np.sum((d[k:]-dbar)*(d[:-k]-dbar))/T
    dm = (dbar/np.sqrt(lrv/T)) * np.sqrt((T + 1 - 2*h + h*(h-1)/T)/T)
    return float(dm), float(2*(1 - stats.t.cdf(abs(dm), df=T-1)))
print('Metrics + DM ready.')

Metrics + DM ready.


In [4]:
def select_arima_order(y_train):
    best = {'aic': np.inf, 'order': (1,1,1)}
    for p,d,q in itertools.product(range(4), range(3), range(4)):
        try:
            r = SARIMAX(y_train, order=(p,d,q),
                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            if r.aic < best['aic']: best = {'aic': r.aic, 'order': (p,d,q)}
        except Exception: pass
    return best['order'], best['aic']

def select_sarima_order(y_train, s=12):
    best = {'aic': np.inf, 'order': (1,1,1), 'sorder': (0,1,1,s)}
    for (p,d,q),(P,D,Q) in itertools.product(
            itertools.product(range(3), range(2), range(3)),
            itertools.product(range(2), range(2), range(2))):
        try:
            r = SARIMAX(y_train, order=(p,d,q), seasonal_order=(P,D,Q,s),
                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            if r.aic < best['aic']: best = {'aic': r.aic, 'order': (p,d,q), 'sorder': (P,D,Q,s)}
        except Exception: pass
    return best['order'], best['sorder'], best['aic']

def rolling_one_step(y_full, order, sorder, test_start, test_end):
    y_train = y_full.loc[:TRAIN_END]; y_test = y_full.loc[test_start:test_end]
    fit = SARIMAX(y_train, order=order, seasonal_order=sorder,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    ext = fit.append(y_test, refit=False)
    pred = ext.get_prediction(start=y_test.index[0]).predicted_mean
    return y_test.values - pred.values, y_test.values
print('Selection + rolling forecaster ready.')

Selection + rolling forecaster ready.


In [5]:
IMF_BASE = 'https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/CPI/~'
TEST_URL = f'{IMF_BASE}/BRA.CPI._T.IX.M?c[TIME_PERIOD]=ge:2020-M01'
try:
    r = requests.get(TEST_URL, headers={'Accept':'text/csv'}, timeout=60)
    print('HTTP', r.status_code)
    if r.status_code == 200 and len(r.text) > 100:
        t = pd.read_csv(io.StringIO(r.text), low_memory=False)
        print('OK — columns:', list(t.columns)[:12])
        print(t.head(3).to_string())
        print('\n>>> API WORKS. Use Cell 5a.')
    else:
        print('Unexpected response:'); print(r.text[:400])
        print('\n>>> Use Cell 5b (manual CSV).')
except Exception as ex:
    print('FAILED:', type(ex).__name__, ex)
    print('\n>>> Use Cell 5b (manual CSV).')

HTTP 200
OK — columns: ['STRUCTURE[;]', 'STRUCTURE_ID', 'ACTION', 'COUNTRY', 'INDEX_TYPE', 'COICOP_1999', 'TYPE_OF_TRANSFORMATION', 'FREQUENCY', 'TIME_PERIOD', 'OBS_VALUE', 'SCALE', 'PRECISION']
  STRUCTURE[;]        STRUCTURE_ID ACTION COUNTRY INDEX_TYPE COICOP_1999 TYPE_OF_TRANSFORMATION FREQUENCY TIME_PERIOD  OBS_VALUE  SCALE  PRECISION  DECIMALS_DISPLAYED  REPORTING_PERIOD_TYPE  TRANSFORMATION  UNIT DERIVATION_TYPE OVERLAP REFERENCE_PERIOD COMMON_REFERENCE_PERIOD  STATUS  IFS_FLAG
0     dataflow  IMF.STA:CPI(5.0.0)      R     BRA        CPI          _T                     IX         M    2020-M01    5331.42      0        NaN                 NaN                    NaN             NaN   NaN               O      OL          1993M12                 1993M12     NaN      True
1     dataflow  IMF.STA:CPI(5.0.0)      R     BRA        CPI          _T                     IX         M    2020-M02    5344.75      0        NaN                 NaN                    NaN             NaN   NaN    

In [6]:
cpi_data = {}

def _parse_imf_csv(text):
    df = pd.read_csv(io.StringIO(text), low_memory=False)
    cols = {c.upper(): c for c in df.columns}
    ccol = cols.get('COUNTRY') or cols.get('REF_AREA')
    tcol = cols.get('TIME_PERIOD'); vcol = cols.get('OBS_VALUE')
    if not (ccol and tcol and vcol):
        raise ValueError(f'unexpected columns: {list(df.columns)}')
    df = df[[ccol, tcol, vcol]].copy()
    df.columns = ['country','period','value']
    df['date'] = pd.to_datetime(df['period'].astype(str).str.replace('M','', regex=False),
                                format='%Y-%m', errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.dropna(subset=['date','value'])

key = '+'.join(COUNTRIES.keys()) + '.CPI._T.IX.M'
url = f'{IMF_BASE}/{key}?c[TIME_PERIOD]=ge:2000-M01'
print('GET', url)
try:
    r = requests.get(url, headers={'Accept':'text/csv'}, timeout=180)
    r.raise_for_status()
    tidy = _parse_imf_csv(r.text)
    for iso3, name in COUNTRIES.items():
        sub = tidy[tidy['country'].astype(str).str.upper() == iso3]
        if sub.empty:
            print(f'  [{iso3}] {name:12s} not returned — use Cell 5b'); continue
        s = sub.set_index('date')['value'].sort_index()
        s = s[~s.index.duplicated(keep='first')].asfreq('MS').dropna()
        cpi_data[iso3] = s
        print(f'  [{iso3}] {name:12s} {s.index.min().date()} -> {s.index.max().date()}  n={s.size}')
except Exception as ex:
    print('API fetch failed:', type(ex).__name__, ex)
print('\nLoaded:', list(cpi_data.keys()))

GET https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/CPI/~/BGD+IND+PAK+LKA+IDN+PHL.CPI._T.IX.M?c[TIME_PERIOD]=ge:2000-M01
  [BGD] Bangladesh   2010-01-01 -> 2026-06-01  n=198
  [IND] India        2000-01-01 -> 2026-06-01  n=318
  [PAK] Pakistan     2008-07-01 -> 2026-06-01  n=216
  [LKA] Sri Lanka    2010-01-01 -> 2026-06-01  n=198
  [IDN] Indonesia    2000-01-01 -> 2026-06-01  n=318
  [PHL] Philippines  2010-01-01 -> 2026-06-01  n=198

Loaded: ['BGD', 'IND', 'PAK', 'LKA', 'IDN', 'PHL']


In [8]:
BD_CSV = f'{FOLDER}/bangladesh_MASTER_dataset.csv'
try:
    _bd = pd.read_csv(BD_CSV, index_col=0, parse_dates=True)
    s = _bd['CPI'].loc['2000-01-01':TEST_END].asfreq('MS').dropna()
    cpi_data['BGD'] = s
    print(f'Bangladesh control loaded: {s.index.min().date()} -> {s.index.max().date()} n={s.size}')
except Exception as ex:
    print(f'Could not load {BD_CSV}: {ex}')

Bangladesh control loaded: 2000-01-01 -> 2026-04-01 n=316


In [10]:
rows = []
for iso3, name in COUNTRIES.items():
    if iso3 not in cpi_data:
        rows.append({'ISO3':iso3,'country':name,'status':'MISSING','n_train':0,'n_test':0}); continue
    y = cpi_data[iso3].dropna()
    t_end = min(pd.Timestamp(TEST_END), y.index.max())
    ntr = int(y.loc[:TRAIN_END].size); nte = int(y.loc[TEST_START:t_end].size)
    ok = (ntr >= MIN_TRAIN) and (nte >= MIN_TEST)
    rows.append({'ISO3':iso3,'country':name,'status':'OK' if ok else 'SKIP',
                 'start':str(y.index.min().date()), 'end':str(y.index.max().date()),
                 'n_train':ntr, 'n_test':nte})
cov = pd.DataFrame(rows)
print(cov.to_string(index=False))
runnable = cov[cov.status=='OK']['ISO3'].tolist()
print(f'\nWill run: {runnable}')
if len(runnable) < 3:
    print('WARNING: fewer than 3 countries — external validity needs 3-4 minimum.')

ISO3     country status      start        end  n_train  n_test
 BGD  Bangladesh     OK 2000-01-01 2026-04-01      252      64
 IND       India     OK 2000-01-01 2026-06-01      252      64
 PAK    Pakistan     OK 2008-07-01 2026-06-01      150      64
 LKA   Sri Lanka     OK 2010-01-01 2026-06-01      132      64
 IDN   Indonesia     OK 2000-01-01 2026-06-01      252      64
 PHL Philippines     OK 2010-01-01 2026-06-01      132      64

Will run: ['BGD', 'IND', 'PAK', 'LKA', 'IDN', 'PHL']


In [16]:
LSTM_OK = False
if RUN_LSTM:
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dropout, Dense
        from tensorflow.keras.callbacks import EarlyStopping
        LSTM_OK = True; print('TensorFlow', tf.__version__)
    except Exception as ex:
        print('TensorFlow unavailable:', ex)

def make_windows(arr, look=12):
    X, y = [], []
    for i in range(look, len(arr)):
        X.append(arr[i-look:i]); y.append(arr[i])
    return np.array(X), np.array(y)

def lstm_one_step(y_full, test_start, test_end, seed, look=12):
    tf.keras.backend.clear_session(); tf.random.set_seed(seed); np.random.seed(seed)
    y_train = y_full.loc[:TRAIN_END]; y_test = y_full.loc[test_start:test_end]
    full = y_full.loc[:test_end]
    mu, sd = float(y_train.mean()), float(y_train.std())
    scaled = ((full - mu)/sd).values
    p0 = full.index.get_loc(y_test.index[0])
    Xtr, ytr = make_windows(scaled[:p0], look); Xtr = Xtr.reshape(*Xtr.shape, 1)
    m = Sequential([LSTM(64, input_shape=(look,1)), Dropout(0.2),
                    Dense(32, activation='relu'), Dense(1)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    m.fit(Xtr, ytr, epochs=200, batch_size=16, verbose=0, validation_split=0.1,
          callbacks=[EarlyStopping(patience=25, restore_best_weights=True)])
    W = np.array([scaled[p0+i-look:p0+i] for i in range(len(y_test))]).reshape(len(y_test), look, 1)
    preds = m.predict(W, verbose=0).flatten()*sd + mu
    return y_test.values - preds, y_test.values
print('LSTM ready' if LSTM_OK else 'LSTM disabled')

TensorFlow 2.20.0
LSTM ready


In [17]:
results = {}
for iso3, name in COUNTRIES.items():
    if iso3 not in cpi_data:
        print(f'--- {name}: skipped (no data)'); continue
    y = cpi_data[iso3].dropna()
    t_end = min(pd.Timestamp(TEST_END), y.index.max())
    ntr, nte = y.loc[:TRAIN_END].size, y.loc[TEST_START:t_end].size
    if ntr < MIN_TRAIN or nte < MIN_TEST:
        print(f'--- {name}: skipped (train {ntr}, test {nte})'); continue
    t0 = time.time()
    print(f'=== {name} ({iso3}) | train {ntr} | test {nte} ===')
    R = {'name':name,'n_train':int(ntr),'n_test':int(nte),'test_end':str(t_end.date())}

    e_naive, actual = naive_error_vectors(y, TEST_START, t_end)
    for k,v in e_naive.items(): R[f'{k}_RMSE'] = round(rmse(v),4)

    ao,_ = select_arima_order(y.loc[:TRAIN_END])
    e_ar,_ = rolling_one_step(y, ao, (0,0,0,0), TEST_START, t_end)
    R['ARIMA_order']=str(ao); R['ARIMA_RMSE']=round(rmse(e_ar),4)

    so,sso,_ = select_sarima_order(y.loc[:TRAIN_END])
    e_sar,_ = rolling_one_step(y, so, sso, TEST_START, t_end)
    R['SARIMA_order']=f'{so}x{sso}'
    R['SARIMA_RMSE']=round(rmse(e_sar),4); R['SARIMA_MAE']=round(mae(e_sar),4)
    R['SARIMA_MAPE']=round(mape(e_sar, actual),4)

    d,p = dm_test(e_sar, e_naive['RW+drift']); R['DM_SARIMA_vs_RWdrift']=round(d,3); R['p_SARIMA_vs_RWdrift']=round(p,4)
    d,p = dm_test(e_sar, e_ar);               R['DM_SARIMA_vs_ARIMA']=round(d,3);   R['p_SARIMA_vs_ARIMA']=round(p,4)

    if RUN_LSTM and LSTM_OK:
        rl=[]; e42=None
        for sdv in LSTM_SEEDS:
            e_l,_ = lstm_one_step(y, TEST_START, t_end, sdv); rl.append(rmse(e_l))
            if sdv==42: e42=e_l
        R['LSTM_RMSE_mean']=round(float(np.mean(rl)),4); R['LSTM_RMSE_sd']=round(float(np.std(rl,ddof=1)),4)
        R['LSTM_RMSE_min']=round(float(np.min(rl)),4);   R['LSTM_RMSE_max']=round(float(np.max(rl)),4)
        d,p = dm_test(e_sar, e42); R['DM_SARIMA_vs_LSTM']=round(d,3); R['p_SARIMA_vs_LSTM']=round(p,4)

    cand={'SARIMA':R['SARIMA_RMSE'],'ARIMA':R['ARIMA_RMSE'],'RW+drift':R['RW+drift_RMSE']}
    if 'LSTM_RMSE_mean' in R: cand['LSTM']=R['LSTM_RMSE_mean']
    R['winner']=min(cand,key=cand.get)
    results[iso3]=R
    print(f"    SARIMA {R['SARIMA_RMSE']} | ARIMA {R['ARIMA_RMSE']} | RW+drift {R['RW+drift_RMSE']}"
          + (f" | LSTM {R.get('LSTM_RMSE_mean')}" if 'LSTM_RMSE_mean' in R else '')
          + f" -> {R['winner']}  ({time.time()-t0:.0f}s)")
print('\nRun complete.')

=== Bangladesh (BGD) | train 252 | test 64 ===


    SARIMA 1.3053 | ARIMA 2.6216 | RW+drift 2.7781 | LSTM 27.1766 -> SARIMA  (227s)
=== India (IND) | train 252 | test 64 ===
    SARIMA 0.4799 | ARIMA 0.5571 | RW+drift 0.6163 | LSTM 6.3644 -> SARIMA  (216s)
=== Pakistan (PAK) | train 150 | test 64 ===
    SARIMA 3.2861 | ARIMA 3.2445 | RW+drift 3.6466 | LSTM 73.8845 -> ARIMA  (134s)
=== Sri Lanka (LKA) | train 132 | test 64 ===
    SARIMA 6.5135 | ARIMA 4.4413 | RW+drift 4.3352 | LSTM 66.9865 -> RW+drift  (105s)
=== Indonesia (IDN) | train 252 | test 64 ===
    SARIMA 0.414 | ARIMA 0.3993 | RW+drift 0.4003 | LSTM 3.1084 -> ARIMA  (178s)
=== Philippines (PHL) | train 132 | test 64 ===
    SARIMA 0.6315 | ARIMA 0.6624 | RW+drift 0.6851 | LSTM 11.6393 -> SARIMA  (132s)

Run complete.


In [19]:
if 'BGD' in results:
    bd = results['BGD']
    ok_rmse  = abs(bd['SARIMA_RMSE'] - 1.3053) < 0.01
    ok_order = bd['SARIMA_order'] == '(2, 1, 2)x(0, 1, 1, 12)'
    print(f"CONTROL  SARIMA {bd['SARIMA_order']}  RMSE {bd['SARIMA_RMSE']}  MAE {bd['SARIMA_MAE']}  MAPE {bd['SARIMA_MAPE']}")
    print('  RMSE  matches 1.3053 :', 'PASS' if ok_rmse else 'FAIL')
    print('  order matches paper  :', 'PASS' if ok_order else f"DIFFERS ({bd['SARIMA_order']})")
    print('  RW+drift 2.7781      :', 'PASS' if abs(bd['RW+drift_RMSE']-2.7781)<0.001 else 'FAIL')
    print('\n>>> Control PASSED — other countries are trustworthy.' if (ok_rmse and ok_order)
          else '\n>>> Control did not pass; do not use the other countries yet.')

df_res = pd.DataFrame(results).T
cols = ['name','n_train','n_test','SARIMA_order','SARIMA_RMSE','ARIMA_RMSE','RW+drift_RMSE']
if RUN_LSTM: cols += ['LSTM_RMSE_mean','LSTM_RMSE_sd']
cols += ['DM_SARIMA_vs_RWdrift','p_SARIMA_vs_RWdrift','winner']
cols = [c for c in cols if c in df_res.columns]
print('\n' + df_res[cols].to_string())

wins = [r['winner'] for r in results.values()]; n = len(wins)
print(f"\nHEADLINE: SARIMA champion in {sum(w=='SARIMA' for w in wins)}/{n} economies; "
      f"LSTM in {sum(w=='LSTM' for w in wins)}/{n}; RW+drift in {sum(w=='RW+drift' for w in wins)}/{n}.")
sig = sum(1 for r in results.values() if r.get('p_SARIMA_vs_RWdrift', 1) < 0.05
          and r.get('DM_SARIMA_vs_RWdrift', 0) < 0)
print(f"SARIMA significantly beats RW+drift (DM, 5%) in {sig}/{n} economies.")

CONTROL  SARIMA (2, 1, 2)x(0, 1, 1, 12)  RMSE 1.3053  MAE 0.937  MAPE 0.3838
  RMSE  matches 1.3053 : PASS
  order matches paper  : PASS
  RW+drift 2.7781      : PASS

>>> Control PASSED — other countries are trustworthy.

            name n_train n_test             SARIMA_order SARIMA_RMSE ARIMA_RMSE RW+drift_RMSE LSTM_RMSE_mean LSTM_RMSE_sd DM_SARIMA_vs_RWdrift p_SARIMA_vs_RWdrift    winner
BGD   Bangladesh     252     64  (2, 1, 2)x(0, 1, 1, 12)      1.3053     2.6216        2.7781        27.1766       4.7022               -4.829                 0.0    SARIMA
IND        India     252     64  (1, 0, 1)x(0, 1, 1, 12)      0.4799     0.5571        0.6163         6.3644       2.9663               -2.527               0.014    SARIMA
PAK     Pakistan     150     64  (0, 1, 2)x(0, 1, 1, 12)      3.2861     3.2445        3.6466        73.8845       4.9819               -2.058              0.0438     ARIMA
LKA    Sri Lanka     132     64  (1, 1, 2)x(0, 1, 1, 12)      6.5135     4.4413      

In [20]:
payload = {'protocol': {'train_end':TRAIN_END,'test_start':TEST_START,'test_end':TEST_END,
             'method':'fit-once then append(refit=False); AIC grids identical to Bangladesh Session 1',
             'source':'IMF Data API SDMX 3.0, dataflow IMF.STA/CPI, key {ISO3}.CPI._T.IX.M'},
           'results': results}
json.dump(payload, open(f'{FOLDER}/move2_results.json','w'), indent=1)
pd.DataFrame(results).T.to_csv(f'{FOLDER}/move2_results.csv')
print('Saved move2_results.json and move2_results.csv')
try:
    from google.colab import files
    files.download('move2_results.json'); files.download('move2_results.csv')
except Exception:
    print('(Download manually from the file panel.)')

Saved move2_results.json and move2_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

def make_windows(arr, look=12):
    X, y = [], []
    for i in range(look, len(arr)):
        X.append(arr[i-look:i]); y.append(arr[i])
    return np.array(X), np.array(y)

def lstm_one_step(y_full, test_start, test_end, seed, look=12):
    """Univariate LSTM on the MONTHLY CHANGE, reconstructed to levels.
       Scaler fitted on training changes only. Strictly causal."""
    tf.keras.backend.clear_session(); tf.random.set_seed(seed); np.random.seed(seed)
    y_test = y_full.loc[test_start:test_end]
    full   = y_full.loc[:test_end]
    d_full = full.diff()
    d_train = d_full.loc[:TRAIN_END].dropna()
    mu, sd = float(d_train.mean()), float(d_train.std())
    scaled = ((d_full - mu)/sd).values
    p0 = full.index.get_loc(y_test.index[0])
    Xtr, ytr = make_windows(scaled[1:p0], look)
    Xtr = Xtr.reshape(*Xtr.shape, 1)
    m = Sequential([LSTM(64, input_shape=(look,1)), Dropout(0.2),
                    Dense(32, activation='relu'), Dense(1)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    m.fit(Xtr, ytr, epochs=200, batch_size=16, verbose=0, validation_split=0.1,
          callbacks=[EarlyStopping(patience=25, restore_best_weights=True)])
    W = np.array([scaled[p0+i-look:p0+i] for i in range(len(y_test))]).reshape(len(y_test), look, 1)
    dhat = m.predict(W, verbose=0).flatten()*sd + mu
    preds = full.values[p0-1:p0-1+len(y_test)] + dhat
    return y_test.values - preds, y_test.values

print('LSTM fixed: target = monthly change')

LSTM fixed: target = monthly change


In [22]:
import ast
for iso3 in list(results.keys()):
    R = results[iso3]
    y = cpi_data[iso3].dropna()
    t_end = min(pd.Timestamp(TEST_END), y.index.max())
    o, sp = R['SARIMA_order'].split(')x(')
    e_sar, _ = rolling_one_step(y, ast.literal_eval(o+')'), ast.literal_eval('('+sp),
                                TEST_START, t_end)
    rl = []; e42 = None
    for sdv in LSTM_SEEDS:
        e_l, _ = lstm_one_step(y, TEST_START, t_end, sdv); rl.append(rmse(e_l))
        if sdv == 42: e42 = e_l
    R['LSTM_RMSE_mean']=round(float(np.mean(rl)),4); R['LSTM_RMSE_sd']=round(float(np.std(rl,ddof=1)),4)
    R['LSTM_RMSE_min']=round(float(np.min(rl)),4);   R['LSTM_RMSE_max']=round(float(np.max(rl)),4)
    d,p = dm_test(e_sar, e42); R['DM_SARIMA_vs_LSTM']=round(d,3); R['p_SARIMA_vs_LSTM']=round(p,4)
    cand = {'SARIMA':R['SARIMA_RMSE'],'ARIMA':R['ARIMA_RMSE'],
            'RW+drift':R['RW+drift_RMSE'],'LSTM':R['LSTM_RMSE_mean']}
    R['winner'] = min(cand, key=cand.get)
    print(f"  {R['name']:12s} LSTM {R['LSTM_RMSE_mean']:8.4f} (sd {R['LSTM_RMSE_sd']:.4f}) -> {R['winner']}")
print('LSTM patch complete.')

  Bangladesh   LSTM   3.1533 (sd 0.3055) -> SARIMA
  India        LSTM   0.5243 (sd 0.0077) -> SARIMA
  Pakistan     LSTM   3.6954 (sd 0.0457) -> ARIMA
  Sri Lanka    LSTM   4.9662 (sd 0.1465) -> RW+drift
  Indonesia    LSTM   0.6189 (sd 0.2376) -> ARIMA
  Philippines  LSTM   0.6487 (sd 0.0118) -> SARIMA
LSTM patch complete.


In [23]:
if 'BGD' in results:
    bd = results['BGD']
    ok_rmse  = abs(bd['SARIMA_RMSE'] - 1.3053) < 0.01
    ok_order = bd['SARIMA_order'] == '(2, 1, 2)x(0, 1, 1, 12)'
    print(f"CONTROL  SARIMA {bd['SARIMA_order']}  RMSE {bd['SARIMA_RMSE']}  MAE {bd['SARIMA_MAE']}  MAPE {bd['SARIMA_MAPE']}")
    print('  RMSE  matches 1.3053 :', 'PASS' if ok_rmse else 'FAIL')
    print('  order matches paper  :', 'PASS' if ok_order else f"DIFFERS ({bd['SARIMA_order']})")
    print('  RW+drift 2.7781      :', 'PASS' if abs(bd['RW+drift_RMSE']-2.7781)<0.001 else 'FAIL')
    print('\n>>> Control PASSED — other countries are trustworthy.' if (ok_rmse and ok_order)
          else '\n>>> Control did not pass; do not use the other countries yet.')

df_res = pd.DataFrame(results).T
cols = ['name','n_train','n_test','SARIMA_order','SARIMA_RMSE','ARIMA_RMSE','RW+drift_RMSE']
if RUN_LSTM: cols += ['LSTM_RMSE_mean','LSTM_RMSE_sd']
cols += ['DM_SARIMA_vs_RWdrift','p_SARIMA_vs_RWdrift','winner']
cols = [c for c in cols if c in df_res.columns]
print('\n' + df_res[cols].to_string())

wins = [r['winner'] for r in results.values()]; n = len(wins)
print(f"\nHEADLINE: SARIMA champion in {sum(w=='SARIMA' for w in wins)}/{n} economies; "
      f"LSTM in {sum(w=='LSTM' for w in wins)}/{n}; RW+drift in {sum(w=='RW+drift' for w in wins)}/{n}.")
sig = sum(1 for r in results.values() if r.get('p_SARIMA_vs_RWdrift', 1) < 0.05
          and r.get('DM_SARIMA_vs_RWdrift', 0) < 0)
print(f"SARIMA significantly beats RW+drift (DM, 5%) in {sig}/{n} economies.")

CONTROL  SARIMA (2, 1, 2)x(0, 1, 1, 12)  RMSE 1.3053  MAE 0.937  MAPE 0.3838
  RMSE  matches 1.3053 : PASS
  order matches paper  : PASS
  RW+drift 2.7781      : PASS

>>> Control PASSED — other countries are trustworthy.

            name n_train n_test             SARIMA_order SARIMA_RMSE ARIMA_RMSE RW+drift_RMSE LSTM_RMSE_mean LSTM_RMSE_sd DM_SARIMA_vs_RWdrift p_SARIMA_vs_RWdrift    winner
BGD   Bangladesh     252     64  (2, 1, 2)x(0, 1, 1, 12)      1.3053     2.6216        2.7781         3.1533       0.3055               -4.829                 0.0    SARIMA
IND        India     252     64  (1, 0, 1)x(0, 1, 1, 12)      0.4799     0.5571        0.6163         0.5243       0.0077               -2.527               0.014    SARIMA
PAK     Pakistan     150     64  (0, 1, 2)x(0, 1, 1, 12)      3.2861     3.2445        3.6466         3.6954       0.0457               -2.058              0.0438     ARIMA
LKA    Sri Lanka     132     64  (1, 1, 2)x(0, 1, 1, 12)      6.5135     4.4413      

In [24]:
df = pd.DataFrame(results).T
for c in ['SARIMA_RMSE','ARIMA_RMSE','RW+drift_RMSE','LSTM_RMSE_mean']:
    df[c] = df[c].astype(float)
df['SARIMA/RWd']  = (df['SARIMA_RMSE']/df['RW+drift_RMSE']).round(3)
df['LSTM/RWd']    = (df['LSTM_RMSE_mean']/df['RW+drift_RMSE']).round(3)
df['LSTM/SARIMA'] = (df['LSTM_RMSE_mean']/df['SARIMA_RMSE']).round(3)
show = ['name','n_train','SARIMA_RMSE','ARIMA_RMSE','RW+drift_RMSE','LSTM_RMSE_mean',
        'LSTM_RMSE_sd','SARIMA/RWd','LSTM/RWd','LSTM/SARIMA',
        'DM_SARIMA_vs_LSTM','p_SARIMA_vs_LSTM','winner']
print(df[[c for c in show if c in df.columns]].to_string())

w = df['winner'].tolist(); n = len(w)
print("\nSCOREBOARD")
for k in ['SARIMA','ARIMA','RW+drift','LSTM']:
    print(f"  {k:9s} champion in {sum(x==k for x in w)}/{n}")
print(f"  statistical family (SARIMA or ARIMA): {sum(x in ('SARIMA','ARIMA') for x in w)}/{n}")
print(f"  LSTM beats RW+drift in {(df['LSTM/RWd']<1).sum()}/{n}")
print(f"  SARIMA beats LSTM in  {(df['LSTM/SARIMA']>1).sum()}/{n}")

json.dump({'protocol':{'train_end':TRAIN_END,'test_start':TEST_START,'test_end':TEST_END,
  'lstm':'univariate on monthly change (dCPI), train-only scaler, 10 seeds, reconstructed to levels'},
  'results':results}, open(f'{FOLDER}/move2_results.json','w'), indent=1)
df.to_csv(f'{FOLDER}/move2_results.csv')
print(f"\nSaved to {FOLDER}")

            name n_train  SARIMA_RMSE  ARIMA_RMSE  RW+drift_RMSE  LSTM_RMSE_mean LSTM_RMSE_sd  SARIMA/RWd  LSTM/RWd  LSTM/SARIMA DM_SARIMA_vs_LSTM p_SARIMA_vs_LSTM    winner
BGD   Bangladesh     252       1.3053      2.6216         2.7781          3.1533       0.3055       0.470     1.135        2.416            -5.295              0.0    SARIMA
IND        India     252       0.4799      0.5571         0.6163          0.5243       0.0077       0.779     0.851        1.093             -1.41           0.1633    SARIMA
PAK     Pakistan     150       3.2861      3.2445         3.6466          3.6954       0.0457       0.901     1.013        1.125            -2.149           0.0355     ARIMA
LKA    Sri Lanka     132       6.5135      4.4413         4.3352          4.9662       0.1465       1.502     1.146        0.762             3.986           0.0002  RW+drift
IDN    Indonesia     252       0.4140      0.3993         0.4003          0.6189       0.2376       1.034     1.546        1.495  

In [25]:
payload = {'protocol': {'train_end':TRAIN_END,'test_start':TEST_START,'test_end':TEST_END,
             'method':'fit-once then append(refit=False); AIC grids identical to Bangladesh Session 1',
             'source':'IMF Data API SDMX 3.0, dataflow IMF.STA/CPI, key {ISO3}.CPI._T.IX.M'},
           'results': results}
json.dump(payload, open(f'{FOLDER}/move2_results.json','w'), indent=1)
pd.DataFrame(results).T.to_csv(f'{FOLDER}/move2_results.csv')
print('Saved move2_results.json and move2_results.csv')
try:
    from google.colab import files
    files.download('move2_results.json'); files.download('move2_results.csv')
except Exception:
    print('(Download manually from the file panel.)')

Saved move2_results.json and move2_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
print(f"{'country':13s}{'max MoM %':>11s}{'when':>12s}{'train YoY sd':>14s}{'test YoY sd':>13s}{'peak YoY %':>12s}")
for iso3 in results:
    y = cpi_data[iso3].dropna()
    mom = y.pct_change()*100
    yoy = y.pct_change(12)*100
    tr_yoy = yoy.loc[:TRAIN_END].dropna(); te_yoy = yoy.loc[TEST_START:].dropna()
    j = mom.abs().idxmax()
    print(f"{results[iso3]['name']:13s}{mom.loc[j]:11.2f}{str(j.date()):>12s}"
          f"{tr_yoy.std():14.2f}{te_yoy.std():13.2f}{te_yoy.max():12.2f}")
print("\nRebasing splice = isolated MoM jump far larger than neighbours.")
print("Regime break    = test YoY sd >> train YoY sd, with a high peak.")

country        max MoM %        when  train YoY sd  test YoY sd  peak YoY %
Bangladesh          4.20  2008-06-01          2.25         1.85       11.67
India               4.58  2009-07-01          2.92         1.66        7.79
Pakistan            6.34  2022-06-01          3.60        10.26       37.97
Sri Lanka          10.49  2022-04-01          2.49        22.87       72.54
Indonesia           8.71  2005-10-01          3.64         1.35        5.95
Philippines         2.63  2026-04-01          1.51         2.01        8.68

Rebasing splice = isolated MoM jump far larger than neighbours.
Regime break    = test YoY sd >> train YoY sd, with a high peak.
